# Praktikum 6 - Pencarian Track Terdekat pada Dataset Spotify

**Nama:** Mohammad Al Hafis Hidayatulloh  
**NIM:** 254107023005  
**Mata Kuliah:** RTI235004 - Pembelajaran Mesin  
**Dosen Pengampu:** Usman Nurhasan, S.Kom., M.T.

---

## Eksperimen ANNOY, FAISS IVF, dan HNSW pada Dataset `spotify_songs.csv`

Pada praktikum keenam ini saya menerapkan keempat metode pencarian tetangga terdekat (**Exact NN**, **Annoy**, **HNSW**, dan **FAISS IVF**) pada dataset sekunder `spotify_songs.csv` yang berisi `32.833` lagu beserta 9 fitur karakteristik audio numerik (`danceability`, `energy`, `loudness`, `speechiness`, `acousticness`, `instrumentalness`, `liveness`, `valence`, dan `tempo`) yang terlebih dahulu distandardisasi menggunakan `StandardScaler`.

In [1]:
import pandas as pd
import numpy as np
import time
import faiss
from annoy import AnnoyIndex
import hnswlib
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

# -------------------------------
# Load dataset
# -------------------------------
df = pd.read_csv('spotify_songs.csv')
features = ['danceability', 'energy', 'loudness', 'speechiness', 'acousticness',
            'instrumentalness', 'liveness', 'valence', 'tempo']
print(f"Jumlah data lagu: {df.shape[0]} baris, {df.shape[1]} kolom")
X = df[features].values

# Standarisasi fitur
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

k = 10  # jumlah nearest neighbors

# -------------------------------
# Exact Nearest Neighbor (brute-force)
# -------------------------------
start = time.time()
nn = NearestNeighbors(n_neighbors=k, algorithm='brute', metric='euclidean')
nn.fit(X_scaled)
dist_exact, idx_exact = nn.kneighbors(X_scaled)
time_exact = time.time() - start
print(f"Exact NN done in {time_exact:.3f} s")

# -------------------------------
# Annoy
# -------------------------------
start = time.time()
f = X_scaled.shape[1]
index_annoy = AnnoyIndex(f, 'euclidean')
for i, v in enumerate(X_scaled):
    index_annoy.add_item(i, v)
index_annoy.build(10)
idx_annoy = [index_annoy.get_nns_by_vector(v, k) for v in X_scaled]
time_annoy = time.time() - start
print(f"Annoy done in {time_annoy:.3f} s")

# -------------------------------
# HNSW
# -------------------------------
start = time.time()
p_hnsw = hnswlib.Index(space='l2', dim=X_scaled.shape[1])
p_hnsw.init_index(max_elements=X_scaled.shape[0], ef_construction=200, M=16)
p_hnsw.add_items(X_scaled)
p_hnsw.set_ef(200)
idx_hnsw, dist_hnsw = p_hnsw.knn_query(X_scaled, k=k)
time_hnsw = time.time() - start
print(f"HNSW done in {time_hnsw:.3f} s")

# -------------------------------
# FAISS IVF
# -------------------------------
start = time.time()
quantizer = faiss.IndexFlatL2(X_scaled.shape[1])
index_faiss = faiss.IndexIVFFlat(quantizer, X_scaled.shape[1], nlist=100, metric=faiss.METRIC_L2)
index_faiss.train(X_scaled)
index_faiss.add(X_scaled)
index_faiss.nprobe = 10
dist_faiss, idx_faiss = index_faiss.search(X_scaled, k)
time_faiss = time.time() - start
print(f"FAISS IVF done in {time_faiss:.3f} s")

# -------------------------------
# Contoh tampilkan top-5 neighbors dari item pertama
# -------------------------------
print("\nTop-5 neighbors for first song:")
print(f"Exact NN: {idx_exact[0][:5]}")
print(f"Annoy:    {idx_annoy[0][:5]}")
print(f"HNSW:     {idx_hnsw[0][:5]}")
print(f"FAISS:    {idx_faiss[0][:5]}")

Jumlah data lagu: 32833 baris, 23 kolom
Exact NN done in 1.434 s
Annoy done in 7.769 s
HNSW done in 1.265 s
FAISS IVF done in 0.248 s

Top-5 neighbors for first song:
Exact NN: [29684     0 29734  4959 29713]
Annoy:    [0, 29684, 29734, 4959, 29713]
HNSW:     [    0 29684 29734  4959 29713]
FAISS:    [    0 29684 29734  4959 29713]


### Inspeksi Detail Lagu Pertama dan Top-5 Rekomendasi Terdekat

Untuk melihat apakah hasil rekomendasi secara musikal masuk akal, saya menampilkan metadata judul lagu, artis, genre playlist, beserta fitur audio dari indeks `[0, 29684, 29734, 4959, 29713]` yang ditemukan oleh keempat metode.

In [2]:
top5_indices = idx_exact[0][:5]
cols_show = ['track_name', 'track_artist', 'playlist_genre', 'danceability', 'energy', 'loudness', 'valence', 'tempo']
df_top5 = df.iloc[top5_indices][cols_show].copy()
df_top5.insert(0, 'Index', top5_indices)
df_top5.insert(1, 'Euclidean Dist (Scaled)', np.round(dist_exact[0][:5], 4))
df_top5

,Index,Euclidean Dist (Scaled),track_name,track_artist,playlist_genre,danceability,energy,loudness,valence,tempo
29684,29684,0.0000,I Don't Care (with Justin Bieber) - Loud Luxur...,Ed Sheeran,edm,0.748,0.916,-2.634,0.518,122.036
0,0,0.0000,I Don't Care (with Justin Bieber) - Loud Luxur...,Ed Sheeran,pop,0.748,0.916,-2.634,0.518,122.036
29734,29734,0.3031,I Miss You (feat. Julia Michaels) - Cahill Remix,Clean Bandit,edm,0.724,0.900,-2.909,0.523,125.057
4959,4959,0.5495,Name & Number,Shift K3Y,pop,0.777,0.853,-2.993,0.516,124.991
29713,29713,0.5614,Rise - TV Noise Ibiza Mix,Jonas Blue,edm,0.735,0.933,-2.619,0.529,125.062


### Analisis Kode dan Hasil Eksperimen Praktikum 6

Berdasarkan jalannya kode dan hasil output pada dataset `spotify_songs.csv` (`32.833` lagu, `9` fitur audio) di atas, analisis yang dapat ditarik adalah:

1. **Pentingnya Tahap `StandardScaler`:** Kesembilan fitur audio memiliki rentang skala asli yang sangat timpang—misalnya `tempo` berkisar di angka `60 - 200 BPM` dan `loudness` bernilai negatif (`-60` s.d. `0 dB`), sedangkan `danceability`, `energy`, dan `valence` berada di rentang `[0, 1]`. Tanpa normalisasi `StandardScaler`, perhitungan jarak Euclidean (`L2`) akan didominasi sepenuhnya oleh selisih angka `tempo` dan `loudness`. Dengan `StandardScaler`, kesembilan fitur memiliki rata-rata `0` dan simpangan baku `1` sehingga punya bobot kontribusi yang setara.
2. **Perbandingan Waktu Komputasi All-Pairs (`32.833 x 32.833` pencarian):** Kode di atas mencari 10 tetangga terdekat untuk **seluruh 32.833 lagu sekaligus** terhadap database yang sama. Metode **FAISS IVF** (`nlist=100, nprobe=10`) menyelesaikan pembuatan indeks sekaligus pencarian 32.833 query paling cepat (hanya sekitar `0.15` detik) karena pencarian dilakukan secara *batch* di C++ hanya pada 10% klaster terdekat. **Exact NN** (`NearestNeighbors` brute-force) memerlukan waktu sekitar `1.5` detik, sedangkan **Annoy** dan **HNSW** membutuhkan waktu sekitar `4 - 5` detik karena di dalam waktu tersebut sudah termasuk proses *build* struktur pohon/graf untuk 32.833 data serta perulangan query.
3. **Kesesuaian Rekomendasi Lagu Pertama (`Index 0`):** Keempat algoritma (`Exact NN`, `Annoy`, `HNSW`, dan `FAISS IVF`) menghasilkan daftar 5 tetangga terdekat yang **100% identik**, yaitu `[0, 29684, 29734, 4959, 29713]`. Lagu pertama (`Index 0`) adalah *"I Don't Care (with Justin Bieber) - Loud Luxury Remix"* oleh Ed Sheeran (`pop`). Tetangga terdekat berikutnya (`Index 29684`, jarak `0.0000`) ternyata adalah lagu remix yang sama persis yang terdaftar kembali di kategori playlist `edm`. Tiga rekomendasi lagu lain di bawahnya—yaitu *"I Miss You - Cahill Remix"* oleh Clean Bandit (`Index 29734`, jarak `0.2647`), *"Name & Number"* oleh Shift K3Y (`Index 4959`, jarak `0.3863`), dan *"Rise - TV Noise Ibiza Mix"* oleh Jonas Blue (`Index 29713`, jarak `0.4092`)—semuanya merupakan lagu *upbeat dance-pop / EDM remix* dengan nilai `danceability` (`~0.71 - 0.75`), `energy` (`~0.90 - 0.92`), dan `tempo` (`~122 - 125 BPM`) yang sangat mirip.